In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, classification_report

import pandas as pnd


<h1 style="color: red;">Section 1: Data</h1>

<h2>1) Préparation de données</h2>

In [2]:
dataset =pnd.read_csv('diabetes.csv')#import
X = np.array(dataset.drop(columns=['Outcome'])) #features
y = np.array(dataset['Outcome']) #target
#spilt data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=23)


<h1 style="color: red;">Section 2: Neural network avec tensorflow</h1>

In [3]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

from tensorflow.keras.optimizers import Adam

I0000 00:00:1778439899.781765  133428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778439899.806573  133428 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778439902.563053  133428 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778439908.060809  133428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

<h2>2) Modèle de réseau de neurones</h2>

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 1. Définition du modèle (0 couche cachée, 1 neurone de sortie)
model_nn = Sequential()

# input_shape=(8,) car il y a 8 caractéristiques (features) dans le dataset diabète
# activation='sigmoid' est OBLIGATOIRE pour ramener le résultat entre 0 et 1 (probabilité)
model_nn.add(Dense(units=1, input_shape=(X_train.shape[1],), activation='sigmoid'))

# 2. Compilation du modèle
# loss='binary_crossentropy' est la fonction d'erreur standard pour la classification binaire
# metrics=['accuracy'] permet d'afficher le taux de bonnes réponses pendant l'entraînement
model_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Entraînement
history = model_nn.fit(X_train, y_train, epochs=150, validation_split=0.2)

/workspaces/ML_basics/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1778439908.966137  133428 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4358 - loss: 13.8866 - val_accuracy: 0.4228 - val_loss: 12.6516
Epoch 2/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4481 - loss: 13.3404 - val_accuracy: 0.4472 - val_loss: 12.1287
Epoch 3/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4664 - loss: 12.9984 - val_accuracy: 0.4390 - val_loss: 11.7064
Epoch 4/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4847 - loss: 12.6857 - val_accuracy: 0.4309 - val_loss: 11.4565
Epoch 5/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4868 - loss: 12.4227 - val_accuracy: 0.4472 - val_loss: 11.2545
Epoch 6/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4888 - loss: 12.1427 - val_accuracy: 0.4472 - val_loss: 11.0571
Epoch 7/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4827 - loss: 11.8787 - val_accuracy: 0.4390 - val_loss: 10.8656
Epoch 8/150
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4868 - loss: 11.6226 - val_accu

<h2>3) Prédiction en utilisant le modèle</h2>

In [5]:
yhat_nn=model_nn.predict(X_test)

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


In [6]:
yhat_nn=yhat_nn.flatten()

In [7]:
W_nn, bias_nn = model_nn.layers[0].get_weights()

# Afficher les poids et biais
print("Poids :", W_nn.flatten())
print("Biais :", bias_nn)

Poids : [ 0.23667154  0.01921578 -0.04595896 -0.00410378  0.00044613  0.05086325
 -0.2786668  -0.03827417]
Biais : [-0.73981065]


<h2>4) Evaluation du modèle</h2>

In [8]:
# 1. Faire les prédictions
yhat_prob = model_nn.predict(X_test)

# Les prédictions sont des probabilités (ex: 0.8). Il faut les transformer en classes (0 ou 1)
# Si probabilité > 0.5, on classe à 1, sinon à 0.
yhat_classes = (yhat_prob > 0.5).astype(int)

# 2. Calcul des métriques de performance
print("Matrice de confusion :\n", confusion_matrix(y_test, yhat_classes))
print("\nAccuracy :", accuracy_score(y_test, yhat_classes))
print("Recall   :", recall_score(y_test, yhat_classes))
print("F1-Score :", f1_score(y_test, yhat_classes))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Matrice de confusion :
 [[85 16]
 [21 32]]

Accuracy : 0.7597402597402597
Recall   : 0.6037735849056604
F1-Score : 0.6336633663366337


<h1>From scratch</h1>


<h2>Modèle de régression logistic from scratch avec utilisation des matrices</h2>


In [9]:
# 0. Définition de la fonction Sigmoïde (ajoutez-la juste avant la boucle si elle n'y est pas)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 1. Initialisation des paramètres
# Nous avons 8 colonnes dans X_train, donc W doit avoir une taille de (8, 1)
W = np.zeros((X_train.shape[1], 1)) 
b = 0.0

# Hyperparamètres
learning_rate = 0.001 
epochs = 2000
y_train = y_train.reshape(-1, 1) 
n = len(X_train)

# 2. Boucle d'entraînement
for epoch in range(epochs):
    # Prédiction : X * W + b passé dans la sigmoïde
    z = X_train @ W + b
    y_pred = sigmoid(z)  # Remplace le ? (Shape: n, 1)
    
    # Erreur (différence entre prédiction et réalité)
    error = y_pred - y_train  # Remplace le ? (Shape: n, 1)

    # Calcul des gradients (X transposé multiplié par l'erreur)
    dW = (1/n) * (X_train.T @ error)  # Remplace le ?
    db = (1/n) * np.sum(error)

    # Mise à jour des paramètres
    W -= learning_rate * dW  # Remplace le ?
    b -= learning_rate * db

<h2>Évaluation du modèle manuel</h2>

In [10]:
# Prédiction sur le test set
z_test = X_test @ W + b
y_prob_test = sigmoid(z_test)
y_pred_manual = (y_prob_test > 0.5).astype(int)

print("--- Modèle Manuel ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_manual):.4f}")

--- Modèle Manuel ---
Accuracy : 0.6429


<h2>Comparaison avec Scikit-Learn</h2>

In [11]:
from sklearn.linear_model import LogisticRegression

# Entraînement du modèle standard
sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X_train, y_train.ravel())

# Prédiction
y_pred_sk = sk_model.predict(X_test)

print("--- Modèle Scikit-Learn ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_sk):.4f}")

# Comparaison finale
print(f"\nLes modèles sont-ils proches ? Différence d'accuracy : {abs(accuracy_score(y_test, y_pred_manual) - accuracy_score(y_test, y_pred_sk)):.4f}")

--- Modèle Scikit-Learn ---
Accuracy : 0.8052

Les modèles sont-ils proches ? Différence d'accuracy : 0.1623
